# Experimento 03 — Random Forest

Compara V1–V4 y realiza RandomizedSearchCV moderado sólo sobre train para la mejor variante base.

Sólo se utilizan train y validation. Test no se carga.

In [1]:
from pathlib import Path
import json,time
from datetime import datetime,timezone
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score,balanced_accuracy_score,classification_report,confusion_matrix,f1_score,precision_score,recall_score
ROOT=Path.cwd().parent if Path.cwd().name=="experimentos" else Path.cwd()
DATA=ROOT/"data/processed/modelado/dataset_modelado_nivel_actividad_firms.parquet"
SPLITS=ROOT/"data/processed/modelado/splits";OUT=ROOT/"results/experimentos_parciales";OUT.mkdir(parents=True,exist_ok=True)
datos=pd.read_parquet(DATA);datos.fecha_inicio_semana=pd.to_datetime(datos.fecha_inicio_semana)
KEY=["departamento","fecha_inicio_semana"];TARGET="nivel_actividad_firms";ORDER=["Sin actividad","Bajo","Moderado","Alto"];RS=42
def subset(name):
 k=pd.read_parquet(SPLITS/f"{name}_keys.parquet");k.fecha_inicio_semana=pd.to_datetime(k.fecha_inicio_semana)
 return datos.merge(k,on=KEY,how="inner",validate="one_to_one")
train=subset("train");validation=subset("validation")
assert len(train)==5546 and len(validation)==1188
CAT=["departamento","mes"]
MET=["temperature_2m_max_mean_weekly","relative_humidity_2m_min_mean_weekly","wind_speed_10m_max_mean_weekly","precipitation_sum_weekly"]
LAGS=[f"cantidad_detecciones_lag{i}" for i in range(1,5)]
ROLL=["promedio_detecciones_ultimas_4_semanas","max_detecciones_ultimas_4_semanas","semanas_con_actividad_ultimas_4"]
VARIANTS={"V1":CAT+MET,"V2":CAT+MET+LAGS,"V3":CAT+MET+ROLL,"V4":CAT+MET+LAGS+ROLL}
PROHIBITED={"cantidad_detecciones","fecha_inicio_semana","anio","numero_semana",TARGET,"nivel_actividad_firms_codigo"}
assert all(PROHIBITED.isdisjoint(v) for v in VARIANTS.values())
def metrics(y,p):
 r=classification_report(y,p,labels=ORDER,output_dict=True,zero_division=0)
 return {"accuracy_validation":accuracy_score(y,p),"balanced_accuracy_validation":balanced_accuracy_score(y,p),"precision_macro_validation":precision_score(y,p,average="macro",zero_division=0),"recall_macro_validation":recall_score(y,p,average="macro",zero_division=0),"f1_macro_validation":f1_score(y,p,average="macro",zero_division=0),"f1_weighted_validation":f1_score(y,p,average="weighted",zero_division=0),**{f"precision_{x.lower().replace(' ','_')}":r[x]["precision"] for x in ORDER},**{f"recall_{x.lower().replace(' ','_')}":r[x]["recall"] for x in ORDER},**{f"f1_{x.lower().replace(' ','_')}":r[x]["f1-score"] for x in ORDER}}
def row(eid,model,variant,params,pred,seconds,note):return {"experiment_id":eid,"modelo":model,"variante_variables":variant,"hiperparametros":json.dumps(params,sort_keys=True,ensure_ascii=False),**metrics(validation[TARGET],pred),"duracion_segundos":seconds,"fecha_ejecucion":datetime.now(timezone.utc).isoformat(),"notebook_origen":NOTEBOOK,"observaciones":note}

from scipy.stats import randint,uniform
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
NOTEBOOK="Experimento_03_RandomForest.ipynb"
def build(cols,**params):
 cat=[x for x in cols if x in CAT];num=[x for x in cols if x not in CAT]
 pre=ColumnTransformer([("cat",OneHotEncoder(handle_unknown="ignore"),cat),("num","passthrough",num)])
 defaults=dict(n_estimators=300,min_samples_leaf=2,class_weight="balanced_subsample",random_state=RS,n_jobs=-1);defaults.update(params)
 return Pipeline([("pre",pre),("model",RandomForestClassifier(**defaults))])

In [2]:
rows=[];base_predictions={}
for v,cols in VARIANTS.items():
 t=time.perf_counter();mod=build(cols).fit(train[cols],train[TARGET]);pred=mod.predict(validation[cols]);elapsed=time.perf_counter()-t;base_predictions[v]=pred
 rows.append(row(f"rf_base_{v.lower()}","RandomForest",v,{"n_estimators":300,"min_samples_leaf":2,"class_weight":"balanced_subsample","random_state":RS,"n_jobs":-1},pred,elapsed,"Configuración base"))
base=pd.DataFrame(rows);base[["variante_variables","f1_macro_validation","balanced_accuracy_validation","f1_bajo","f1_moderado","f1_alto"]].round(4)
for v,pred_v in base_predictions.items():
 print(f"Matriz de confusión validation - {v}")
 display(pd.DataFrame(confusion_matrix(validation[TARGET],pred_v,labels=ORDER),index=ORDER,columns=ORDER))

Matriz de confusión validation - V1


,Sin actividad,Bajo,Moderado,Alto
Sin actividad,636,42,50,17
Bajo,119,21,34,11
Moderado,96,23,53,18
Alto,27,2,17,22


Matriz de confusión validation - V2


,Sin actividad,Bajo,Moderado,Alto
Sin actividad,647,24,61,13
Bajo,130,14,32,9
Moderado,117,18,45,10
Alto,29,2,21,16


Matriz de confusión validation - V3


,Sin actividad,Bajo,Moderado,Alto
Sin actividad,643,25,66,11
Bajo,119,18,37,11
Moderado,111,20,46,13
Alto,30,0,18,20


Matriz de confusión validation - V4


,Sin actividad,Bajo,Moderado,Alto
Sin actividad,651,24,60,10
Bajo,129,15,34,7
Moderado,120,18,41,11
Alto,29,1,17,21


In [3]:
best_variant=base.sort_values(["f1_macro_validation","balanced_accuracy_validation"],ascending=False).iloc[0].variante_variables;cols=VARIANTS[best_variant]
search=RandomizedSearchCV(build(cols),{"model__n_estimators":[200,300,500],"model__max_depth":[None,10,20],"model__min_samples_leaf":[1,2,5],"model__max_features":["sqrt",0.5],"model__class_weight":["balanced","balanced_subsample"]},n_iter=12,scoring="f1_macro",cv=3,random_state=RS,n_jobs=-1,refit=True,return_train_score=False)
t=time.perf_counter();search.fit(train[cols],train[TARGET]);elapsed=time.perf_counter()-t;pred=search.predict(validation[cols]);best_params={k.replace("model__",""):v for k,v in search.best_params_.items()};best_params.update({"random_state":RS,"n_jobs":-1})
tuned=pd.DataFrame([row(f"rf_tuned_{best_variant.lower()}","RandomForest",best_variant,best_params,pred,elapsed,f"RandomizedSearchCV 12 iteraciones, cv=3; best_cv={search.best_score_:.4f}")]);results=pd.concat([base,tuned],ignore_index=True);results.sort_values("f1_macro_validation",ascending=False).round(4)

,experiment_id,modelo,variante_variables,hiperparametros,accuracy_validation,balanced_accuracy_validation,precision_macro_validation,recall_macro_validation,f1_macro_validation,f1_weighted_validation,...,recall_moderado,recall_alto,f1_sin_actividad,f1_bajo,f1_moderado,f1_alto,duracion_segundos,fecha_ejecucion,notebook_origen,observaciones
4,rf_tuned_v1,RandomForest,V1,"{""class_weight"": ""balanced_subsample"", ""max_de...",0.5749,0.4257,0.3966,0.4257,0.4052,0.5793,...,0.2842,0.4265,0.7568,0.2621,0.2865,0.3152,61.0142,2026-09-30T19:16:22.661019+00:00,Experimento_03_RandomForest.ipynb,"RandomizedSearchCV 12 iteraciones, cv=3; best_..."
0,rf_base_v1,RandomForest,V1,"{""class_weight"": ""balanced_subsample"", ""min_sa...",0.6162,0.3924,0.4077,0.3924,0.3923,0.5832,...,0.2789,0.3235,0.7837,0.1538,0.3081,0.3235,2.2153,2026-09-30T19:15:14.981155+00:00,Experimento_03_RandomForest.ipynb,Configuración base
2,rf_base_v3,RandomForest,V3,"{""class_weight"": ""balanced_subsample"", ""min_sa...",0.6120,0.3742,0.4092,0.3742,0.3771,0.5718,...,0.2421,0.2941,0.7803,0.1452,0.2577,0.3252,2.3663,2026-09-30T19:15:19.838643+00:00,Experimento_03_RandomForest.ipynb,Configuración base
3,rf_base_v4,RandomForest,V4,"{""class_weight"": ""balanced_subsample"", ""min_sa...",0.6128,0.3699,0.4144,0.3699,0.3750,0.5659,...,0.2158,0.3088,0.7778,0.1235,0.2398,0.3590,1.5167,2026-09-30T19:15:21.409295+00:00,Experimento_03_RandomForest.ipynb,Configuración base
1,rf_base_v2,RandomForest,V2,"{""class_weight"": ""balanced_subsample"", ""min_sa...",0.6077,0.3541,0.3897,0.3541,0.3562,0.5615,...,0.2368,0.2353,0.7758,0.1152,0.2579,0.2759,2.3657,2026-09-30T19:15:17.423284+00:00,Experimento_03_RandomForest.ipynb,Configuración base


In [4]:
display(pd.DataFrame(confusion_matrix(validation[TARGET],pred,labels=ORDER),index=ORDER,columns=ORDER));display(pd.DataFrame(classification_report(validation[TARGET],pred,labels=ORDER,output_dict=True,zero_division=0)).T.loc[ORDER].round(4))
pi=permutation_importance(search.best_estimator_,validation[cols],validation[TARGET],scoring="f1_macro",n_repeats=8,random_state=RS,n_jobs=-1);imp=pd.DataFrame({"variable":cols,"media":pi.importances_mean,"std":pi.importances_std}).sort_values("media",ascending=False);imp.round(4)

,Sin actividad,Bajo,Moderado,Alto
Sin actividad,554,74,79,38
Bajo,84,46,39,16
Moderado,65,38,54,33
Alto,16,8,15,29


,precision,recall,f1-score,support
Sin actividad,0.7705,0.7436,0.7568,745.0
Bajo,0.2771,0.2486,0.2621,185.0
Moderado,0.2888,0.2842,0.2865,190.0
Alto,0.2500,0.4265,0.3152,68.0


,variable,media,std
1,mes,0.0710,0.0087
0,departamento,0.0667,0.0103
3,relative_humidity_2m_min_mean_weekly,0.0660,0.0080
2,temperature_2m_max_mean_weekly,0.0436,0.0083
5,precipitation_sum_weekly,0.0188,0.0089
4,wind_speed_10m_max_mean_weekly,0.0162,0.0060


In [5]:
errors=validation[KEY+[TARGET]].copy();errors["pred"]=pred;code={x:i for i,x in enumerate(ORDER)};errors["distancia"]=(errors[TARGET].map(code)-errors.pred.map(code)).abs();display(errors.distancia.value_counts(normalize=True).sort_index());results.to_csv(OUT/"03_random_forest.csv",index=False)

distancia
0    0.574916
1    0.238215
2    0.141414
3    0.045455
Name: proportion, dtype: float64